# Youtube Download

In [ ]:
import yt_dlp

def download_video(song_name):
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'outtmpl': 'input_video.%(ext)s',
        'default_search': 'ytsearch1:',  # 搜索并取第一个结果
        'noplaylist': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([song_name])
    return "input_video.mp4"

# Music track extraction
## fast-whisper
句子不完整

In [ ]:
import os
import subprocess
from faster_whisper import WhisperModel

def extract_and_transcribe(video_path):
    audio_path = "temp_audio.wav"
    
    # --- 步骤 1: 使用 FFmpeg 提取标准音频 ---
    print("正在提取音轨...")
    subprocess.run([
        'ffmpeg', '-i', video_path, 
        '-ar', '16000', '-ac', '1', '-c:a', 'pcm_s16le', 
        audio_path, '-y'
    ], check=True)

    # --- 步骤 2: 加载 Whisper 模型 ---
    # Mac M1/M2/M3 用户建议使用 cpu 模式（faster-whisper 对 MPS 支持在持续更新，CPU 已经够快）
    model_size = "medium" # 嘻哈建议 medium 起步，large-v3 效果最好但慢一点
    model = WhisperModel(model_size, device="cpu", compute_type="int8")

    print(f"正在识别歌词 (模型: {model_size})...")
    segments, info = model.transcribe(
        audio_path,
        # 1. 简化的 Prompt：只给关键词，不给完整句子
        initial_prompt="Rap, Hip-hop, lyrics, slang, A$AP, Travis Scott.", 
        
        # 2. 提高采样门槛：如果 AI 不确定，就不要乱说话
        beam_size=5,
        best_of=5,
        
        # 3. 关键：设置静音过滤和压缩比限制
        vad_filter=True,  # 确保开启静音过滤
        vad_parameters=dict(min_silence_duration_ms=500), # 超过0.5秒没声音就跳过
        
        # 4. 幻听控制：如果这一段重复率太高，丢弃它
        compression_ratio_threshold=2.4, 
        no_speech_threshold=0.6,
        
        word_timestamps=True
    )

    # --- 步骤 3: 格式化为 SRT 结构 ---
    results = []
    for segment in segments:
        results.append({
            'start': segment.start,
            'end': segment.end,
            'text': segment.text.strip()
        })
        print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")

    # 清理临时文件
    os.remove(audio_path)
    return results

# 使用示例
# lyrics = extract_and_transcribe("your_downloaded_video.mp4")

## WhisperX
句子完整但很长

In [ ]:
import whisperx
import gc

def process_mv_with_whisperx(video_path, device="cpu"):
    # 1. 加载模型 (针对 Mac M1/M2/M3 优化)
    model = whisperx.load_model("medium", device, compute_type="int8")
    
    # 2. 直接传视频路径，whisperx 会自动提取音频流进行转录
    print(f"开始转录视频内容: {video_path}")
    result = model.transcribe(video_path, batch_size=16) 
    
    # 3. 对齐 (这是解决你“断句碎裂”的关键)
    # 它会加载 wav2vec2 模型，精准对齐每个单词的时间点
    model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
    aligned_result = whisperx.align(result["segments"], model_a, metadata, video_path, device, return_char_alignments=False)
    
    # 4. 语义合并 (解决歌词碎裂的逻辑)
    # WhisperX 提供了更加连贯的 segments，但我们还可以手动根据停顿（Gap）进一步优化
    final_segments = merge_broken_sentences(aligned_result["segments"])
    
    return final_segments

def merge_broken_sentences(segments, min_pause=0.8):
    """
    如果两段歌词之间的停顿小于 min_pause 秒，且上一段没有结束标点，则合并。
    """
    merged = []
    if not segments: return merged
    
    curr = segments[0]
    for i in range(1, len(segments)):
        nxt = segments[i]
        # 判断：如果两段间隔很短（比如 0.5秒内），大概率是同一句歌词换气
        gap = nxt['start'] - curr['end']
        
        if gap < min_pause:
            curr['text'] = curr['text'].strip() + " " + nxt['text'].strip()
            curr['end'] = nxt['end']
        else:
            merged.append(curr)
            curr = nxt
    merged.append(curr)
    return merged